# Notebook 4: Facebook Prophet Modelling
Prophet handles trend + seasonality + holidays natively. Easier to tune than ARIMA.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from prophet import Prophet
from sklearn.metrics import mean_absolute_error, mean_squared_error

df_agg = pd.read_csv('../data/processed/featured_aggregated_data.csv', parse_dates=['date'])
CUTOFF = pd.Timestamp('2024-06-01')

def mape(actual, predicted):
    return np.mean(np.abs((actual - predicted) / np.maximum(actual, 1))) * 100

# Indian pharma holidays — affects logistics and demand
holidays = pd.DataFrame({
    'holiday': [
        'Diwali', 'Diwali', 'Diwali',
        'Holi',   'Holi',   'Holi',
        'Republic Day', 'Republic Day', 'Republic Day',
    ],
    'ds': pd.to_datetime([
        '2022-10-24', '2023-11-12', '2024-11-01',
        '2022-03-18', '2023-03-08', '2024-03-25',
        '2022-01-26', '2023-01-26', '2024-01-26',
    ]),
    'lower_window': [-2, -2, -2, -1, -1, -1, -1, -1, -1],
    'upper_window': [1,  1,  1,  1,  1,  1,  0,  0,  0 ],
})

print('Holidays defined:', holidays['holiday'].unique().tolist())
products = df_agg['product_name'].unique()
print('Products:', products.tolist())

## 1. Fit Prophet per Product

In [ ]:
prophet_results = {}
fig, axes = plt.subplots(4, 2, figsize=(16, 20))
axes = axes.flatten()

for i, prod in enumerate(products):
    sub = df_agg[df_agg['product_name'] == prod][['date', 'total_units_sold']].copy()
    sub.columns = ['ds', 'y']

    train = sub[sub['ds'] < CUTOFF]
    test  = sub[sub['ds'] >= CUTOFF]

    model = Prophet(
        seasonality_mode='multiplicative',
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        holidays=holidays,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10,
    )
    model.add_seasonality(name='quarterly', period=91.25, fourier_order=5)
    model.fit(train)

    future = model.make_future_dataframe(periods=len(test), freq='MS')
    forecast = model.predict(future)
    forecast_test = forecast[forecast['ds'] >= CUTOFF]

    y_true = test['y'].values
    y_pred = forecast_test['yhat'].values[:len(y_true)]
    y_pred = np.maximum(y_pred, 0)  # no negative units

    mae_val  = mean_absolute_error(y_true, y_pred)
    rmse_val = np.sqrt(mean_squared_error(y_true, y_pred))
    mape_val = mape(y_true, y_pred)
    prophet_results[prod] = {'MAE': round(mae_val, 1), 'RMSE': round(rmse_val, 1), 'MAPE': round(mape_val, 2)}

    # Plot
    ax = axes[i]
    ax.plot(sub['ds'], sub['y'], color='steelblue', label='Actual', linewidth=1.5)
    fc_plot = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
    ax.plot(fc_plot['ds'], fc_plot['yhat'], color='tomato', linestyle='--', label='Prophet', linewidth=2)
    ax.fill_between(fc_plot['ds'], fc_plot['yhat_lower'], fc_plot['yhat_upper'], alpha=0.15, color='tomato')
    ax.axvline(CUTOFF, color='gray', linestyle=':', linewidth=1)
    ax.set_title(f'{prod}  |  MAPE: {mape_val:.1f}%', fontweight='bold')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=30)
    print(f'{prod:<22} MAE={mae_val:.0f}  RMSE={rmse_val:.0f}  MAPE={mape_val:.2f}%')

plt.suptitle('Facebook Prophet Forecasts — All Products', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../reports/figures/12_prophet_forecasts.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Decompose Components (Trend + Seasonality + Holidays)

In [ ]:
# Show decomposition for Escitab-10 (most interesting seasonality)
sub = df_agg[df_agg['product_name'] == 'Escitab-10'][['date', 'total_units_sold']].copy()
sub.columns = ['ds', 'y']

m = Prophet(seasonality_mode='multiplicative', yearly_seasonality=True,
            weekly_seasonality=False, daily_seasonality=False, holidays=holidays)
m.fit(sub)
future = m.make_future_dataframe(periods=6, freq='MS')
fc = m.predict(future)

fig = m.plot_components(fc)
plt.suptitle('Prophet Components — Escitab-10', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/13_prophet_components.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Prophet Evaluation Summary

In [ ]:
results_df = pd.DataFrame(prophet_results).T.reset_index().rename(columns={'index': 'Product'})
results_df = results_df.sort_values('MAPE')
print('=== Prophet Evaluation on 6-Month Holdout ===')
print(results_df.to_string(index=False))
print(f"\nAverage MAPE: {results_df['MAPE'].mean():.2f}%")
results_df.to_csv('../data/processed/prophet_metrics.csv', index=False)